# 02b — Signal density with a known reference (NCE)

An isolated experiment before continuing notebook 02. Train **one signal candidate** with balanced binary cross entropy and logit
\[
\ell_\psi(x)=f_\psi(x)-\log q_\phi(x).
\]
The saved reference flow is frozen. The NN models the signal log density rather than learning both sides of the ratio. With equal class priors the optimal logit is $\log p_S-\log q_\phi$.

We use an integrable parameterization: $f_\psi=\log g + 30\tanh(h_\psi/30)+c$, where $g$ is a diagonal Gaussian fitted only to optimization signal events. The four-layer, 1024-wide SiLU network learns a bounded correction; $c$ is a learned scalar normalization offset. This is also an architectural change from the baseline classifier, so a successful result would not isolate the effect of the known-reference term alone. Finite integrability does **not** guarantee accurate importance-sampling normalization or correct far tails.

Reuse the current TAG, saved signal sample and reference flow. No background model is required. Nothing is installed into the main hybrid model. Candidate files go under `hybrid/signal_nce/known_q_gaussian_v1/`. A GPU and sufficient host RAM for the 20M-event pool are recommended. The full training is substantial; interrupted training resumes from the last completed epoch.

In [ ]:
# Keep the existing run: this experiment has its own output directory.
import os
import sys
import subprocess
from pathlib import Path

TAG = "sb10_exppoly_10m_slowlr_v3_20261001"  # Fresh run: 10M nominal-ratio events per class and gradual LR decay.
RUN_NAME = TAG
USE_DRIVE = True
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = Path("/content/nsbi_calibration_toolkit")
    BRANCH = "ml4hep_school_tutorial"
    if not (REPO / ".git").exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH,
            "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git", str(REPO),
        ], check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
        subprocess.run([
            "git", "-C", str(REPO), "sparse-checkout", "set", "src",
            "workshops/ml4hep_tifr_colab/calibration",
        ], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    SOURCE = REPO / "workshops/ml4hep_tifr_colab/calibration"
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q", "-r",
        str(SOURCE / "requirements.txt"),
    ], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab/calibration_runs")
    else:
        RUN_BASE = Path("/content/calibration_runs")
else:
    # Open the notebook with calibration/ as its working directory.
    SOURCE = Path.cwd().resolve()
    REPO = SOURCE.parents[2]
    RUN_BASE = SOURCE / "runs"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(SOURCE))
os.chdir(SOURCE)
RUN = Path(os.environ.get("CALIBRATION_RUN", RUN_BASE / RUN_NAME)).resolve()
RUN.mkdir(parents=True, exist_ok=True)
print("Source:", SOURCE)
print("TAG:", TAG)
print("Run:", RUN)
if "CALIBRATION_RUN" in os.environ:
    print("CALIBRATION_RUN overrides the tagged path; verify that it points to the intended run.")
print("Use a fresh kernel when switching between the old exercises and calibration.")

In [ ]:
import gc
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.special import logsumexp
from IPython.display import display
from flow_reference import ReferenceFlow
from sampling import read_sample
from signal_nce import SignalNCE, train_signal, log_density

EXPERIMENT = 'known_q_gaussian_v1'
TRAIN = True  # False: load the saved best candidate and rerun diagnostics only.
N_PER_CLASS = 10_000_000  # Pool size, before the 60/15/25 split.
N_DIAGNOSTIC = 1_000_000
N_NORMALIZATION = 1_000_000
DIAGNOSTIC_SEED = 7_202_610
SETTINGS = dict(seed=7202610, epochs=150, batch_size=4096,
                width=1024, layers=4, bound=30., learning_rate=1e-3)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT = RUN / 'hybrid'
OUT = ROOT / 'signal_nce' / EXPERIMENT
OUT.mkdir(parents=True, exist_ok=True)
REFERENCE = ROOT / 'reference.pt'
SAMPLE = RUN / 'samples' / 'signal.parquet'
for path in [REFERENCE, SAMPLE, RUN / 'config.json', ROOT / 'ratios/signal/ensemble.json']:
    if not path.is_file():
        raise FileNotFoundError(f'Required existing artifact missing: {path}')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(2**20), b''):
            h.update(block)
    return h.hexdigest()

PROVENANCE = dict(reference_sha256=sha256_file(REFERENCE),
                  config_sha256=sha256_file(RUN / 'config.json'),
                  trainer_sha256=sha256_file(SOURCE / 'signal_nce.py'),
                  sample_path=str(SAMPLE), sample_bytes=SAMPLE.stat().st_size,
                  sample_mtime_ns=SAMPLE.stat().st_mtime_ns,
                  noise_seed=SETTINGS['seed'] + 101)
flow = ReferenceFlow.load(REFERENCE, DEVICE)
flow.flow.eval()
for parameter in flow.flow.parameters():
    parameter.requires_grad_(False)
print('Device:', DEVICE, '\nCandidate:', OUT)
print('The saved baseline and reference will not be overwritten.')

## Train or resume

10M signal events from the existing `ratio_train` partition and 10M fresh draws from the frozen reference. Each class has 6M optimization, 1.5M validation and 2.5M reserved events. The Gaussian envelope uses only optimization events. Every batch is balanced, so no class-prior correction is required. NAdam starts at $10^{-3}$; StepLR halves it every ten epochs through 150 epochs (about $6.1\times10^{-8}$ in epochs 141–150). Checkpoint selection uses validation BCE only.

The cached arrays include reference log densities, avoiding repeated flow evaluation. Cache metadata must match exactly; use a new experiment name when changing settings or inputs. Rerunning with completed checkpoints skips optimization. The reserved pool is not used for model selection; the comparisons below use independently generated events.

In [ ]:
if TRAIN:
    cache = OUT / 'training_cache'
    cache.mkdir(exist_ok=True)
    cache_identity = dict(provenance=PROVENANCE, n_per_class=N_PER_CLASS)
    marker = cache / 'complete.json'
    paths = {name: cache / f'{name}.npy' for name in ['signal', 'noise', 'logq_signal', 'logq_noise']}
    if marker.exists():
        if json.loads(marker.read_text()) != cache_identity:
            raise ValueError('Cache provenance changed. Choose a new EXPERIMENT.')
        if not all(p.is_file() for p in paths.values()):
            raise FileNotFoundError('Incomplete cache: remove complete.json and rerun this cell.')
    else:
        x = read_sample(RUN, 'signal', partition='ratio_train', n=N_PER_CLASS)
        np.save(paths['signal'], x)
        np.save(paths['logq_signal'], flow.log_prob(x, batch_size=8192))
        del x
        gc.collect()
        x = flow.sample(N_PER_CLASS, PROVENANCE['noise_seed'], batch_size=8192)
        np.save(paths['noise'], x)
        np.save(paths['logq_noise'], flow.log_prob(x, batch_size=8192))
        del x
        gc.collect()
        marker.write_text(json.dumps(cache_identity, indent=2))
    arrays = {k: np.load(p, mmap_mode='r') for k, p in paths.items()}
    candidate, history = train_signal(arrays['signal'], arrays['noise'],
                                      arrays['logq_signal'], arrays['logq_noise'],
                                      OUT, SETTINGS, PROVENANCE, device=DEVICE)
    del arrays
    gc.collect()
else:
    checkpoint = torch.load(OUT / 'best.pt', map_location=DEVICE, weights_only=False)
    identity = checkpoint['identity']
    if (identity['provenance'] != PROVENANCE or identity['settings'] != SETTINGS
            or identity['n_per_class'] != N_PER_CLASS):
        raise ValueError('Saved candidate does not match the requested experiment.')
    candidate = SignalNCE(**checkpoint['architecture']).to(DEVICE)
    candidate.load_state_dict(checkpoint['model'])
    candidate.eval()
    history = json.loads((OUT / 'history.json').read_text())
    print('Loaded best epoch:', checkpoint['epoch'])

history_frame = pd.DataFrame(history)
history_frame.to_csv(OUT / 'history.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history_frame.plot(x='epoch', y=['train_bce', 'val_bce'], ax=axes[0])
history_frame.plot(x='epoch', y='learning_rate', logy=True, ax=axes[1])
fig.tight_layout()
fig.savefig(OUT / 'training.png', dpi=150)
plt.show()

## Fresh comparison: candidate, baseline ensemble and exact oracle

These events are independent of optimization and validation. Raw logits are used for BCE and classifier calibration. Compare paired BCE differences, exact log-ratio errors, and the populated tail bins together. A visually better reliability curve alone does not establish a better density. Approximate pointwise intervals can be unreliable in sparse bins; inspect their counts.

Reports are generated separately: the candidate is **never averaged with the baseline**. The exact simulator density appears for the first time here. It cannot affect checkpoint selection. Once these diagnostics guide development, use a new diagnostic seed for final validation.

In [ ]:
from sampling import sample_process, process_log_density
from utils import RatioEnsemble
from ratio_diagnostics import diagnostic_report

EVAL = OUT / f'diagnostics_seed_{DIAGNOSTIC_SEED}'
EVAL.mkdir(exist_ok=True)
baseline = RatioEnsemble.load(ROOT / 'ratios/signal', device=DEVICE)
x_num = sample_process('signal', N_DIAGNOSTIC, np.random.default_rng(DIAGNOSTIC_SEED))
x_den = flow.sample(N_DIAGNOSTIC, DIAGNOSTIC_SEED + 1, batch_size=8192)
q_num = flow.log_prob(x_num, batch_size=8192).astype(np.float64)
q_den = flow.log_prob(x_den, batch_size=8192).astype(np.float64)
exact_num = process_log_density(x_num, 'signal') - q_num
exact_den = process_log_density(x_den, 'signal') - q_den
candidate_num = log_density(candidate, x_num) - q_num
candidate_den = log_density(candidate, x_den) - q_den
member_num = baseline.member_log_ratios(x_num)
member_den = baseline.member_log_ratios(x_den)
baseline_num = logsumexp(member_num, axis=0) - np.log(len(member_num))
baseline_den = logsumexp(member_den, axis=0) - np.log(len(member_den))
score_edges = np.unique(np.round(np.r_[np.linspace(0, 1, 26), np.arange(.70, .851, .01)], 8))
summaries = []
for label, ln, ld in [('baseline', member_num, member_den),
                      ('candidate', candidate_num[None, :], candidate_den[None, :])]:
    report = diagnostic_report(ln, ld, exact_num, exact_den,
                               title=f'Signal: {label}', score_edges=score_edges)
    report['summary'].to_csv(EVAL / f'{label}_summary.csv', index=False)
    report['bins'].to_csv(EVAL / f'{label}_bins.csv', index=False)
    chosen = report['summary'].query("model == 'Ensemble'").copy()
    chosen['model'] = label
    summaries.append(chosen)
    display(report['summary'])
    for name, fig in report['figures']:
        fig.savefig(EVAL / f'{label}_{name}.png', dpi=150, bbox_inches='tight')
        display(fig)
        plt.close(fig)
comparison = pd.concat(summaries, ignore_index=True)
comparison.to_csv(EVAL / 'comparison.csv', index=False)
display(comparison)

# Same-event pairing is essential: this error bar concerns evaluation statistics only.
delta_num = np.logaddexp(0, -candidate_num) - np.logaddexp(0, -baseline_num)
delta_den = np.logaddexp(0, candidate_den) - np.logaddexp(0, baseline_den)
difference = .5 * (delta_num.mean() + delta_den.mean())
se = .5 * np.sqrt(delta_num.var(ddof=1)/N_DIAGNOSTIC + delta_den.var(ddof=1)/N_DIAGNOSTIC)
paired = dict(candidate_minus_baseline_bce=float(difference), se=float(se),
              low95=float(difference-1.96*se), high95=float(difference+1.96*se))
(EVAL / 'paired_bce.json').write_text(json.dumps(paired, indent=2))
print('Negative favors candidate; this interval excludes training uncertainty:', paired)

## Independent normalization and tail stress check

Estimate $Z=E_q[\exp(f-\log q)]$ on a **separate** reference bank, then check $E_q[r/Z]$ on the diagnostic denominator bank above. The latter was not used to estimate $Z$. Also record ESS, the largest normalized weight and the fraction of total weight carried by the largest 0.1% of events. A small Monte Carlo SE is not proof that unsampled tails are harmless.

The candidate's normalized log density is $f-\log Z$. This optional scalar correction is saved separately and is not applied retroactively to raw classifier calibration. Both candidate and baseline use the same integration bank for this comparison. No candidate is promoted automatically.

In [ ]:
normalization_x = flow.sample(N_NORMALIZATION, DIAGNOSTIC_SEED + 2, batch_size=8192)
normalization_q = flow.log_prob(normalization_x, batch_size=8192).astype(np.float64)
normalization_members = baseline.member_log_ratios(normalization_x)
normalization_logits = {
    'candidate': log_density(candidate, normalization_x) - normalization_q,
    'baseline': logsumexp(normalization_members, axis=0) - np.log(len(normalization_members)),
}
check_logits = {'candidate': candidate_den, 'baseline': baseline_den}
rows = []
for name, values in normalization_logits.items():
    logz = float(logsumexp(values) - np.log(len(values)))
    scaled = np.exp(values - logz)
    relative_se = float(scaled.std(ddof=1) / np.sqrt(len(scaled)))
    check = check_logits[name] - logz
    logmean = logsumexp(check) - np.log(len(check))
    normalized_weights = np.exp(check - logsumexp(check))
    k = max(1, int(np.ceil(.001 * len(check))))
    rows.append(dict(model=name, log_Z=logz, Z_relative_mc_se=relative_se,
                     independent_mean_ratio=float(np.exp(logmean)),
                     independent_mean_ratio_se=float(np.exp(logmean) *
                         np.exp(check-logmean).std(ddof=1) / np.sqrt(len(check))),
                     independent_ess=float(1 / np.square(normalized_weights).sum()),
                     maximum_normalized_ratio=float(np.exp(check.max())),
                     top_0p1_percent_weight_fraction=float(np.partition(normalized_weights, -k)[-k:].sum())))
normalization = pd.DataFrame(rows)
display(normalization)
normalization.to_csv(EVAL / 'normalization.csv', index=False)
(EVAL / 'normalization.json').write_text(json.dumps(dict(
    reference_sha256=PROVENANCE['reference_sha256'],
    checkpoint_sha256=sha256_file(OUT / 'best.pt'),
    integration_seed=DIAGNOSTIC_SEED+2, integration_size=N_NORMALIZATION,
    diagnostic_size=N_DIAGNOSTIC, results=rows), indent=2))
print('Ready for review:', EVAL)
print('Review comparison.csv, paired_bce.json, calibration/exact-error figures and normalization.csv together.')

## Decision after this run

A promising candidate should improve fresh-event BCE and exact log-ratio errors without merely moving the problem into a less populated tail. Check the 0.70–0.85 region, event counts, extreme weights, ESS, and independent normalization. If the signals conflict or depend strongly on the integration seed, we should investigate before replacing the baseline. A single candidate does not measure seed-to-seed stability; an ensemble would be a subsequent experiment.

To rerun diagnostics, set `TRAIN = False`. To repeat an independent diagnostic bank, change `DIAGNOSTIC_SEED`. To train a different configuration, change `EXPERIMENT` so that the existing experiment remains available. Continue notebook 02 only after reviewing this signal comparison.